In [1]:
# Classification Feature Engineering
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_wine

data = load_wine()
x, y = data.data, data.target
df = pd.DataFrame(x)
df['Target'] = y
print(df.head())

# Pemilihan fitur dengan metode: Filter (filter dengan function statistik tanpa melibatkan algoritma machine learning)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
filter_selector = SelectKBest(score_func=chi2, k=2)
x_train_filter = filter_selector.fit_transform(x_train, y_train)
x_test_filter = filter_selector.transform(x_test)
print(f"Fitur yang dipilih dengan filter method: {filter_selector.get_support(indices=True)}")

# Pemilihan fitur dengan metode: Wrapper (disini dengan RFE (Recuresive Feature Elemination))
model = LogisticRegression(solver='lbfgs', max_iter=5000) # abaikan stdout tentang reach iteration limit
rfe_selector = RFE(model, n_features_to_select=2)
x_train_rfe = rfe_selector.fit_transform(x_train, y_train)
x_test_rfe = rfe_selector.transform(x_test)
print("Fitur yang dipilih dengan Wrapper Methods:", rfe_selector.get_support(indices=True))

# Pemilihan fitur dengan metode: Embedded
# Mendapatkan fitur penting dengan random forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(x_train, y_train)
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

treshold = 0.05
important_features_indices = [i for i in range(len(importances)) if importances[i] > treshold]
x_train_important = x_train[:, important_features_indices]
x_test_important = x_test[:, important_features_indices]
print(f"Fitur yang dipilih dengan Embedded Methods: {important_features_indices}:")
for i in important_features_indices:
  print(f"{data.feature_names[i]}: {importances[i]}")

print("Fitur yang dipilih dengan Embedded Methods (di atas ambang batas):")
for i in important_features_indices:
  print(f"{data.feature_names[i]}: {importances[i]}")

print("\nDimensi data pelatihan dengan fitur penting:", x_train_important.shape)
print("Dimensi data pengujian dengan fitur penting:", x_test_important.shape)

       0     1     2     3      4     5     6     7     8     9    10    11  \
0  14.23  1.71  2.43  15.6  127.0  2.80  3.06  0.28  2.29  5.64  1.04  3.92   
1  13.20  1.78  2.14  11.2  100.0  2.65  2.76  0.26  1.28  4.38  1.05  3.40   
2  13.16  2.36  2.67  18.6  101.0  2.80  3.24  0.30  2.81  5.68  1.03  3.17   
3  14.37  1.95  2.50  16.8  113.0  3.85  3.49  0.24  2.18  7.80  0.86  3.45   
4  13.24  2.59  2.87  21.0  118.0  2.80  2.69  0.39  1.82  4.32  1.04  2.93   

       12  Target  
0  1065.0       0  
1  1050.0       0  
2  1185.0       0  
3  1480.0       0  
4   735.0       0  
Fitur yang dipilih dengan filter method: [ 9 12]


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fitur yang dipilih dengan Wrapper Methods: [0 6]
Fitur yang dipilih dengan Embedded Methods: [0, 6, 9, 10, 11, 12]:
alcohol: 0.11239773542143086
flavanoids: 0.20229341635663622
color_intensity: 0.1712021830864957
hue: 0.07089132259413944
od280/od315_of_diluted_wines: 0.1115643167260497
proline: 0.13904586955351153
Fitur yang dipilih dengan Embedded Methods (di atas ambang batas):
alcohol: 0.11239773542143086
flavanoids: 0.20229341635663622
color_intensity: 0.1712021830864957
hue: 0.07089132259413944
od280/od315_of_diluted_wines: 0.1115643167260497
proline: 0.13904586955351153

Dimensi data pelatihan dengan fitur penting: (142, 6)
Dimensi data pengujian dengan fitur penting: (36, 6)


In [ ]:
def evaluate_model(X_train, X_test, y_train, y_test, model):
  model.fit(X_train, y_train)
  accuracy = model.score(X_test, y_test)
  return accuracy

# Model Logistic Regression untuk Filter Methods
logistic_model_filter = LogisticRegression(max_iter=200)
accuracy_filter = evaluate_model(x_train_filter, x_test_filter, y_train, y_test, logistic_model_filter)

# Model Logistic Regression untuk Wrapper Methods
logistic_model_rfe = LogisticRegression(max_iter=200)
accuracy_rfe = evaluate_model(x_train_rfe, x_test_rfe, y_train, y_test, logistic_model_rfe)

# Model Random Forest untuk Embedded Methods
accuracy_rf = evaluate_model(x_train_important, x_test_important, y_train, y_test, rf_model)

print(f"""
Filter Methods: {accuracy_filter}
Wrapper Methods: {accuracy_rfe}
Embedded Methods: {accuracy_rf}
""")